# Create a user

## What it does

Adds a new user with a name and an email.

## What happens to the saved users

It adds one row. The number and the two dates are set by the database. The email is saved in lowercase. The practice user is deleted at the end, so the real users stay the same.

## Feature it uses

`createUser(name, email)`, from `logic/create-user.js`.

## Failures it can show

- Another user has that email → `Email is already used`.
- No name, or only spaces → `Name is empty`.
- An email without `@` → `Email is invalid`.
- The user cannot be saved → `Could not create the user`.

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
try {
  const { default: Table } = await import("cli-table3");
  const { createUser } = await import("./logic/create-user.js");
  const { deleteUser } = await import("./logic/delete-user.js");

  function printUsers(users) {
    const table = new Table({
      head: ["id", "name", "email", "createdAt", "updatedAt"],
      style: { head: ["cyan", "bold"] },
    });
    for (const user of users) {
      table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
    }
    console.log(table.toString());
  }

  // 1. Create a user: a table with its id and dates
  const user = createUser("Practice", `practice-${Date.now()}@x.com`);
  try {
    printUsers([user]);

    // 2. The same email again: "Email is already used"
    try {
      createUser("Other", user.email);
    } catch (error) {
      console.log(error.message);
    }

    // 3. An empty name: "Name is empty"
    try {
      createUser("   ", "empty@x.com");
    } catch (error) {
      console.log(error.message);
    }

    // 4. An email without @: "Email is invalid"
    try {
      createUser("Practice", "practice.x.com");
    } catch (error) {
      console.log(error.message);
    }
  } finally {
    deleteUser(user.id);
  }
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌────┬──────────┬──────────────────────────────┬─────────────────────┬─────────────────────┐
│ id │ name     │ email                        │ createdAt           │ updatedAt           │
├────┼──────────┼──────────────────────────────┼─────────────────────┼─────────────────────┤
│ 22 │ Practice │ practice-1793027113300@x.com │ 2026-10-26 16:05:13 │ 2026-10-26 16:05:13 │
└────┴──────────┴──────────────────────────────┴─────────────────────┴─────────────────────┘
Email is already used
Name is empty
Email is invalid
